# Your first interactive search planner
Use the Python 3.11 environment from the [README](../README.md), start a fresh kernel and choose **Run All**. This example drives the public Augsburg starter with SMP A*. SUMO controls the surrounding traffic.

The search adapter uses the shared **lane-following, constant-speed predictor**, `predict_observations()`, to forecast surrounding vehicles at each replanning step. [Predict traffic](#Predict-traffic) below explains the model and inspects its output.


In [ ]:
from planner.common.scenarios import ASSETS
from planner.common.visualization import animate
from planner.runtime.evaluation import describe_result
from planner.runtime.execution import run_spec
from planner.spec import smp_spec

starter = ASSETS / "scenarios/public/DEU_AugsburgCRC26P-23_16300_I-1.xml"
result = run_spec(starter, smp_spec())
describe_result(result)


## Inspect the drive
A* searches to the goal, executes up to five steps (0.5 s), then replans. Each row below describes a planning call. Final validity checks the goal, dynamics, collisions and road boundaries against simulated traffic.

**SM1 is the final score; lower is better.** A* uses travel time internally. Runs execute in this kernel without an automatic timeout; interrupt a slow run if needed.

<details>
<summary>Search adapter: state compatibility, prediction and validation</summary>

The unchanged search algorithms use **KS/BMW 320i**, 4.508 × 1.610 m, `dt=0.1`,
and five-step primitives. Adapter behavior:

- **Initial state:** never snapped. Off-grid speeds get five-step, constant-acceleration
  transitions to neighboring library speeds, holding steering fixed and checking
  time/collisions. Out-of-range speeds can join the nearest edge only if feasible;
  reversing is unsupported. Steering must match within `1e-6`; on-grid starts use
  ordinary primitives. Executed steering is retained because SUMO drops it from
  `InitialState`. Discontinuities are rejected; longer/general connectors and
  intermediate goals are not provided.
- **Search horizon:** relative time zero to the original remaining deadline, rounded
  up to a primitive boundary. Successor expansion stops there. Current actors use
  lane-following forecasts; stationary actors, static obstacles and road boundaries
  remain. Recorded futures, future departures and SUMO routes are excluded.
- **Execution:** a complete path to the original goal must be found first. Execute
  one primitive, or a shorter goal-reaching prefix, then replan. Convert internal
  rear-axle coordinates to vehicle centers once and restore absolute timestamps.
- **Memory:** competition searches disable visualization-history recording without
  changing search decisions or limiting the frontier. Standalone SMP notebooks keep
  it enabled; custom SMP trees need `RECORD_HISTORY` support for this optimization.

Prefixes check continuity, dynamics, forecast collisions and boundaries; completed
drives are checked against actual simulated traffic and original planning problems.
Prefix and final dynamics checks call the upstream CommonRoad checker directly.
The supplied benchmarks already use local map coordinates; vehicle limits and
numerical tolerances are unchanged.

</details>


In [ ]:
result.diagnostics


## Predict traffic
The supplied predictor follows lanes at current longitudinal speed and preserves lateral offset. It follows unique successors, continues straight at forks or lane ends, and falls back to current speed/heading if lane projection fails. It does not predict braking or lane changes.

A* checks future collisions using these forecasts. SUMO may move actors differently. This inspection reads only the initial observations from the completed run.


In [ ]:
from planner.common.prediction import predict_observations, search_horizon

problem = next(iter(result.planning_problems.planning_problem_dict.values()))
predicted = predict_observations(
    result.scenario, problem.initial_state.time_step, search_horizon(problem)
)
actor = predicted.dynamic_obstacles[0]
[(state.time_step, state.position.round(2))
 for state in actor.prediction.trajectory.state_list[:5]]


Forecast time `0` means **now**. Returned planner trajectories use absolute times; the adapters handle this conversion. See [04](04_reuse_and_evaluate_planners.ipynb#Use-and-improve-the-predictor) to use or replace the predictor.

## Watch the run
The ego is green and the goal is yellow. Use the slider to inspect individual frames.


In [ ]:
animate(result)


## Save the example
Export writes `solution.xml`, simulated `scenario.xml` and `metadata.json` together. This in-memory example is marked `in_memory`; [04](04_reuse_and_evaluate_planners.ipynb) exports from your saved submission source.


In [ ]:
from pathlib import Path
from planner.runtime.artifacts import export_result

root = Path.cwd()
if root.name == "tutorials":
    root = root.parent
export_result(result, root / "outputs" / "search-example")


Try [02: example planners](02_example_planners.ipynb), or continue to [04: your submission](04_reuse_and_evaluate_planners.ipynb).
